In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/cleansteps

/content/drive/MyDrive/cleansteps


In [ ]:
%%writefile experiments/08_bert_relevant_chunks.py

from __future__ import annotations

import json
import random
import inspect
import logging
import shutil
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)


@dataclass
class Config:
    train_path: str = "data/splits/train.csv"
    valid_path: str = "data/splits/valid.csv"
    test_path: str = "data/splits/test.csv"

    output_dir: str = "outputs/08_bert_relevant_chunks"

    text_col: str = "case_text"
    label_col: str = "label"

    model_name: str = "bert-base-uncased"

    max_length: int = 512
    chunk_size: int = 128
    chunk_stride: int = 128
    max_selected_chunks: int = 4

    top_k: int = 5
    random_state: int = 42

    num_train_epochs: int = 8
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 2

    train_batch_size_gpu: int = 8
    eval_batch_size_gpu: int = 8

    save_total_limit: int = 2
    require_gpu: bool = True

    tfidf_max_features: int = 30000
    tfidf_min_df: int = 2
    tfidf_max_df: float = 0.95

    debug_small_run: bool = False
    debug_train_size: int = 80
    debug_valid_size: int = 40
    debug_test_size: int = 40

    @property
    def output_path(self) -> Path:
        path = Path(self.output_dir)
        path.mkdir(parents=True, exist_ok=True)
        return path

    @property
    def checkpoint_path(self) -> Path:
        path = self.output_path / "checkpoints"
        path.mkdir(parents=True, exist_ok=True)
        return path


CFG = Config()


def reset_output_dir(cfg: Config) -> None:
    out = Path(cfg.output_dir)

    if out.exists():
        shutil.rmtree(out)

    out.mkdir(parents=True, exist_ok=True)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, tuple):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    return obj


def set_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


class PrecomputedClinicalDataset(torch.utils.data.Dataset):
    def __init__(self, examples: list[dict]):
        self.examples = examples

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> dict:
        return self.examples[idx]


def load_splits(cfg: Config):
    train_df = pd.read_csv(cfg.train_path)
    valid_df = pd.read_csv(cfg.valid_path)
    test_df = pd.read_csv(cfg.test_path)

    required_cols = [cfg.text_col, cfg.label_col]

    for split_name, df in [
        ("train", train_df),
        ("valid", valid_df),
        ("test", test_df),
    ]:
        missing = [col for col in required_cols if col not in df.columns]

        if missing:
            raise ValueError(f"{split_name} split missing columns: {missing}")

        df.dropna(subset=required_cols, inplace=True)
        df[cfg.text_col] = df[cfg.text_col].astype(str)
        df[cfg.label_col] = df[cfg.label_col].astype(str)

    if cfg.debug_small_run:
        train_df = train_df.sample(
            min(cfg.debug_train_size, len(train_df)),
            random_state=cfg.random_state,
        )

        valid_df = valid_df.sample(
            min(cfg.debug_valid_size, len(valid_df)),
            random_state=cfg.random_state,
        )

        test_df = test_df.sample(
            min(cfg.debug_test_size, len(test_df)),
            random_state=cfg.random_state,
        )

        logger.warning("DEBUG_SMALL_RUN is active.")

    logger.info("Train rows: %d", len(train_df))
    logger.info("Valid rows: %d", len(valid_df))
    logger.info("Test rows: %d", len(test_df))

    return train_df, valid_df, test_df


def build_label_mappings(train_df, valid_df, test_df, cfg: Config):
    labels = sorted(train_df[cfg.label_col].unique().tolist())

    label2id = {
        label: idx
        for idx, label in enumerate(labels)
    }

    id2label = {
        idx: label
        for label, idx in label2id.items()
    }

    for split_name, df in [
        ("valid", valid_df),
        ("test", test_df),
    ]:
        split_labels = set(df[cfg.label_col].unique())
        missing = split_labels - set(labels)

        if missing:
            raise ValueError(
                f"{split_name} contains labels not present in train: {sorted(missing)}"
            )

    logger.info("Number of classes: %d", len(label2id))

    return label2id, id2label


def fit_tfidf_vectorizer(train_df: pd.DataFrame, cfg: Config) -> TfidfVectorizer:
    logger.info("Fitting TF-IDF vectorizer only on train texts...")

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=cfg.tfidf_max_features,
        ngram_range=(1, 2),
        min_df=cfg.tfidf_min_df,
        max_df=cfg.tfidf_max_df,
    )

    vectorizer.fit(train_df[cfg.text_col].astype(str).tolist())

    logger.info("TF-IDF vocabulary size: %d", len(vectorizer.vocabulary_))

    return vectorizer


def make_token_chunks(
    input_ids: list[int],
    chunk_size: int,
    chunk_stride: int,
) -> list[list[int]]:
    chunks = []

    if not input_ids:
        return chunks

    for start in range(0, len(input_ids), chunk_stride):
        chunk = input_ids[start:start + chunk_size]

        if chunk:
            chunks.append(chunk)

        if start + chunk_size >= len(input_ids):
            break

    return chunks


def score_chunks(
    chunks: list[list[int]],
    tokenizer,
    vectorizer: TfidfVectorizer,
) -> np.ndarray:
    if not chunks:
        return np.array([])

    chunk_texts = [
        tokenizer.decode(
            chunk,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        for chunk in chunks
    ]

    chunk_matrix = vectorizer.transform(chunk_texts)

    scores = np.asarray(
        chunk_matrix.sum(axis=1)
    ).ravel()

    return scores


def build_relevant_chunk_examples(
    df: pd.DataFrame,
    labels: list[int],
    tokenizer,
    vectorizer: TfidfVectorizer,
    cfg: Config,
    split_name: str,
):
    logger.info("Building relevant chunk inputs for %s split...", split_name)

    if tokenizer.cls_token_id is None or tokenizer.sep_token_id is None:
        raise ValueError("Tokenizerul nu are CLS sau SEP token.")

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    max_content_tokens = cfg.max_length - 2

    examples = []
    details = []

    texts = df[cfg.text_col].astype(str).tolist()
    raw_labels = df[cfg.label_col].astype(str).tolist()

    for i, text in enumerate(texts):
        tokenized = tokenizer(
            text,
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
        )

        all_ids = tokenized["input_ids"]
        original_content_length = len(all_ids)
        original_input_length = original_content_length + 2

        chunks = make_token_chunks(
            all_ids,
            chunk_size=cfg.chunk_size,
            chunk_stride=cfg.chunk_stride,
        )

        if original_content_length <= max_content_tokens:
            selected_chunk_indices = list(range(len(chunks)))
            selected_scores = []
            selected_content_ids = all_ids
        else:
            scores = score_chunks(
                chunks=chunks,
                tokenizer=tokenizer,
                vectorizer=vectorizer,
            )

            n_selected = min(
                cfg.max_selected_chunks,
                len(chunks),
            )

            ranked_indices = sorted(
                range(len(chunks)),
                key=lambda idx: (-float(scores[idx]), idx),
            )[:n_selected]

            selected_chunk_indices = sorted(ranked_indices)

            selected_scores = [
                float(scores[idx])
                for idx in selected_chunk_indices
            ]

            selected_content_ids = []

            for chunk_idx in selected_chunk_indices:
                selected_content_ids.extend(chunks[chunk_idx])

            selected_content_ids = selected_content_ids[:max_content_tokens]

        input_ids = [cls_id] + selected_content_ids + [sep_id]
        attention_mask = [1] * len(input_ids)
        token_type_ids = [0] * len(input_ids)

        if len(input_ids) > cfg.max_length:
            raise ValueError(
                f"Input length peste max_length: {len(input_ids)} > {cfg.max_length}"
            )

        examples.append({
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "labels": int(labels[i]),
        })

        details.append({
            "split": split_name,
            "row_index": int(i),
            "label": raw_labels[i],
            "original_content_tokens": int(original_content_length),
            "original_input_tokens": int(original_input_length),
            "num_chunks": int(len(chunks)),
            "selected_chunk_indices": ",".join(str(x) for x in selected_chunk_indices),
            "selected_chunk_scores": ",".join(f"{x:.6f}" for x in selected_scores),
            "selected_content_tokens": int(len(selected_content_ids)),
            "selected_input_tokens": int(len(input_ids)),
            "was_over_512": bool(original_input_length > cfg.max_length),
        })

        if (i + 1) % 250 == 0:
            logger.info("%s: processed %d/%d", split_name, i + 1, len(texts))

    details_df = pd.DataFrame(details)

    details_df.to_csv(
        cfg.output_path / f"{split_name}_chunk_selection_details.csv",
        index=False,
    )

    return examples, details_df


def save_chunk_summary(
    train_details: pd.DataFrame,
    valid_details: pd.DataFrame,
    test_details: pd.DataFrame,
    cfg: Config,
):
    all_details = pd.concat(
        [train_details, valid_details, test_details],
        axis=0,
        ignore_index=True,
    )

    summary = {
        "chunk_size": cfg.chunk_size,
        "chunk_stride": cfg.chunk_stride,
        "max_selected_chunks": cfg.max_selected_chunks,
        "max_length": cfg.max_length,
        "count": int(len(all_details)),
        "mean_original_input_tokens": float(all_details["original_input_tokens"].mean()),
        "median_original_input_tokens": float(all_details["original_input_tokens"].median()),
        "p90_original_input_tokens": float(all_details["original_input_tokens"].quantile(0.90)),
        "p95_original_input_tokens": float(all_details["original_input_tokens"].quantile(0.95)),
        "p99_original_input_tokens": float(all_details["original_input_tokens"].quantile(0.99)),
        "max_original_input_tokens": int(all_details["original_input_tokens"].max()),
        "texts_over_512": int(all_details["was_over_512"].sum()),
        "percent_over_512": float(all_details["was_over_512"].mean()),
        "mean_num_chunks": float(all_details["num_chunks"].mean()),
        "median_num_chunks": float(all_details["num_chunks"].median()),
        "max_num_chunks": int(all_details["num_chunks"].max()),
        "mean_selected_input_tokens": float(all_details["selected_input_tokens"].mean()),
        "median_selected_input_tokens": float(all_details["selected_input_tokens"].median()),
    }

    with open(
        cfg.output_path / "chunk_selection_summary.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(make_json_safe(summary), f, indent=4, ensure_ascii=False)

    plt.figure(figsize=(10, 6))
    plt.hist(all_details["original_input_tokens"], bins=50)
    plt.axvline(cfg.max_length, linestyle="--", label=f"max_length={cfg.max_length}")
    plt.title("Original Token Length Distribution")
    plt.xlabel("Number of tokens")
    plt.ylabel("Number of cases")
    plt.legend()
    plt.tight_layout()
    plt.savefig(cfg.output_path / "original_token_length_distribution.png", dpi=300)
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.hist(all_details["num_chunks"], bins=40)
    plt.title("Number of Chunks per Case")
    plt.xlabel("Number of chunks")
    plt.ylabel("Number of cases")
    plt.tight_layout()
    plt.savefig(cfg.output_path / "chunks_per_case_distribution.png", dpi=300)
    plt.close()

    logger.info("Chunk selection summary saved.")


def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)

    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def compute_metrics_builder(cfg: Config, num_labels: int):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred

        if isinstance(logits, tuple):
            logits = logits[0]

        y_pred = np.argmax(logits, axis=1)
        probs = softmax_np(logits)

        return {
            "accuracy": accuracy_score(labels, y_pred),
            "macro_f1": f1_score(
                labels,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                labels,
                y_pred,
                average="weighted",
                zero_division=0,
            ),
            f"top{cfg.top_k}_accuracy": top_k_accuracy_score(
                labels,
                probs,
                k=cfg.top_k,
                labels=np.arange(num_labels),
            ),
        }

    return compute_metrics


def build_training_args(cfg: Config):
    use_gpu = torch.cuda.is_available()

    logger.info("CUDA available: %s", use_gpu)

    if cfg.require_gpu and not use_gpu:
        raise RuntimeError(
            "GPU nu este activ. In Colab mergi la Runtime -> Change runtime type -> T4 GPU."
        )

    args = {
        "output_dir": str(cfg.checkpoint_path),
        "learning_rate": cfg.learning_rate,
        "per_device_train_batch_size": cfg.train_batch_size_gpu,
        "per_device_eval_batch_size": cfg.eval_batch_size_gpu,
        "num_train_epochs": cfg.num_train_epochs,
        "weight_decay": cfg.weight_decay,
        "logging_dir": str(cfg.output_path / "logs"),
        "logging_strategy": "epoch",
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "save_total_limit": cfg.save_total_limit,
        "report_to": "none",
        "seed": cfg.random_state,
        "fp16": use_gpu,
        "dataloader_pin_memory": use_gpu,
    }

    params = inspect.signature(TrainingArguments.__init__).parameters

    if "eval_strategy" in params:
        args["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in params:
        args["evaluation_strategy"] = "epoch"

    args = {
        key: value
        for key, value in args.items()
        if key in params
    }

    return TrainingArguments(**args)


def save_training_history(trainer: Trainer, cfg: Config):
    history = trainer.state.log_history
    history_df = pd.DataFrame(history)

    history_df.to_csv(
        cfg.output_path / "training_log_history.csv",
        index=False,
    )

    train_logs = [
        item
        for item in history
        if "loss" in item and "eval_loss" not in item and "epoch" in item
    ]

    eval_logs = [
        item
        for item in history
        if "eval_loss" in item and "epoch" in item
    ]

    plt.figure(figsize=(10, 6))

    if train_logs:
        plt.plot(
            [item["epoch"] for item in train_logs],
            [item["loss"] for item in train_logs],
            marker="o",
            label="training loss",
        )

    if eval_logs:
        plt.plot(
            [item["epoch"] for item in eval_logs],
            [item["eval_loss"] for item in eval_logs],
            marker="o",
            label="validation loss",
        )

    plt.title("BERT Relevant Chunks Training and Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(cfg.output_path / "train_valid_loss.png", dpi=300)
    plt.close()


def save_test_outputs(trainer, test_dataset, test_df, id2label, cfg: Config):
    prediction_output = trainer.predict(test_dataset)

    logits = prediction_output.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    y_true = prediction_output.label_ids

    probs = softmax_np(logits)
    y_pred = np.argmax(probs, axis=1)

    num_labels = len(id2label)

    metrics = {
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "test_weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
        f"test_top{cfg.top_k}_accuracy": top_k_accuracy_score(
            y_true,
            probs,
            k=cfg.top_k,
            labels=np.arange(num_labels),
        ),
    }

    with open(
        cfg.output_path / "test_metrics.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(make_json_safe(metrics), f, indent=4, ensure_ascii=False)

    topk_indices = np.argsort(probs, axis=1)[:, ::-1][:, :cfg.top_k]
    topk_probs = np.take_along_axis(probs, topk_indices, axis=1)

    rows = []

    for i in range(len(test_df)):
        true_id = int(y_true[i])
        pred_id = int(y_pred[i])

        rows.append({
            "case_text": test_df.iloc[i][cfg.text_col],
            "true_label": id2label[true_id],
            "predicted_label": id2label[pred_id],
            "top1_probability": float(probs[i, pred_id]),
            f"top{cfg.top_k}_labels": " | ".join(
                id2label[int(idx)]
                for idx in topk_indices[i]
            ),
            f"top{cfg.top_k}_probabilities": " | ".join(
                f"{float(prob):.6f}"
                for prob in topk_probs[i]
            ),
            "correct_top1": bool(true_id == pred_id),
            f"correct_top{cfg.top_k}": bool(true_id in topk_indices[i]),
        })

    pd.DataFrame(rows).to_csv(
        cfg.output_path / "test_predictions.csv",
        index=False,
    )

    report = classification_report(
        y_true,
        y_pred,
        target_names=[
            id2label[i]
            for i in range(num_labels)
        ],
        digits=4,
        zero_division=0,
    )

    with open(
        cfg.output_path / "classification_report.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(report)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(num_labels),
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            id2label[i]
            for i in range(num_labels)
        ],
        columns=[
            id2label[i]
            for i in range(num_labels)
        ],
    )

    cm_df.to_csv(cfg.output_path / "confusion_matrix.csv")

    plt.figure(figsize=(18, 16))
    plt.imshow(cm, interpolation="nearest")
    plt.title("BERT Relevant Chunks Confusion Matrix")
    plt.colorbar()
    plt.xticks(
        np.arange(num_labels),
        [id2label[i] for i in range(num_labels)],
        rotation=90,
        fontsize=7,
    )
    plt.yticks(
        np.arange(num_labels),
        [id2label[i] for i in range(num_labels)],
        fontsize=7,
    )
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(cfg.output_path / "confusion_matrix.png", dpi=300)
    plt.close()

    return metrics


def main():
    cfg = CFG

    reset_output_dir(cfg)
    set_reproducibility(cfg.random_state)

    train_df, valid_df, test_df = load_splits(cfg)

    label2id, id2label = build_label_mappings(
        train_df,
        valid_df,
        test_df,
        cfg,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name,
        use_fast=False,
    )

    with open(
        cfg.output_path / "label_mapping.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            make_json_safe({
                "label2id": label2id,
                "id2label": {
                    str(k): v
                    for k, v in id2label.items()
                },
            }),
            f,
            indent=4,
            ensure_ascii=False,
        )

    vectorizer = fit_tfidf_vectorizer(train_df, cfg)

    train_labels = train_df[cfg.label_col].map(label2id).tolist()
    valid_labels = valid_df[cfg.label_col].map(label2id).tolist()
    test_labels = test_df[cfg.label_col].map(label2id).tolist()

    train_examples, train_details = build_relevant_chunk_examples(
        df=train_df,
        labels=train_labels,
        tokenizer=tokenizer,
        vectorizer=vectorizer,
        cfg=cfg,
        split_name="train",
    )

    valid_examples, valid_details = build_relevant_chunk_examples(
        df=valid_df,
        labels=valid_labels,
        tokenizer=tokenizer,
        vectorizer=vectorizer,
        cfg=cfg,
        split_name="valid",
    )

    test_examples, test_details = build_relevant_chunk_examples(
        df=test_df,
        labels=test_labels,
        tokenizer=tokenizer,
        vectorizer=vectorizer,
        cfg=cfg,
        split_name="test",
    )

    save_chunk_summary(
        train_details=train_details,
        valid_details=valid_details,
        test_details=test_details,
        cfg=cfg,
    )

    train_dataset = PrecomputedClinicalDataset(train_examples)
    valid_dataset = PrecomputedClinicalDataset(valid_examples)
    test_dataset = PrecomputedClinicalDataset(test_examples)

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=len(label2id),
        id2label={
            int(k): v
            for k, v in id2label.items()
        },
        label2id=label2id,
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = build_training_args(cfg)

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_dataset,
        "eval_dataset": valid_dataset,
        "data_collator": data_collator,
        "compute_metrics": compute_metrics_builder(
            cfg,
            num_labels=len(label2id),
        ),
        "callbacks": [
            EarlyStoppingCallback(
                early_stopping_patience=cfg.early_stopping_patience,
            )
        ],
    }

    trainer_params = inspect.signature(Trainer.__init__).parameters

    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    logger.info("Starting BERT relevant chunks fine-tuning...")
    trainer.train()

    logger.info("Evaluating best model on validation split...")
    valid_metrics = trainer.evaluate(valid_dataset)

    with open(
        cfg.output_path / "valid_metrics.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(make_json_safe(valid_metrics), f, indent=4, ensure_ascii=False)

    logger.info("Running final evaluation on test split...")
    test_metrics = save_test_outputs(
        trainer=trainer,
        test_dataset=test_dataset,
        test_df=test_df,
        id2label=id2label,
        cfg=cfg,
    )

    save_training_history(trainer, cfg)

    trainer.save_model(cfg.output_path / "best_model")
    tokenizer.save_pretrained(cfg.output_path / "best_model")

    summary = {
        "experiment": "BERT relevant chunks",
        "model_name": cfg.model_name,
        "max_length": cfg.max_length,
        "chunk_size": cfg.chunk_size,
        "chunk_stride": cfg.chunk_stride,
        "max_selected_chunks": cfg.max_selected_chunks,
        "tfidf_fit": "train split only",
        "tfidf_max_features": cfg.tfidf_max_features,
        "tfidf_min_df": cfg.tfidf_min_df,
        "tfidf_max_df": cfg.tfidf_max_df,
        "num_train_epochs": cfg.num_train_epochs,
        "learning_rate": cfg.learning_rate,
        "weight_decay": cfg.weight_decay,
        "early_stopping_patience": cfg.early_stopping_patience,
        "random_state": cfg.random_state,
        "num_classes": len(label2id),
        "train_rows": len(train_df),
        "valid_rows": len(valid_df),
        "test_rows": len(test_df),
        "valid_metrics": valid_metrics,
        "test_metrics": test_metrics,
    }

    with open(
        cfg.output_path / "experiment_summary.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(make_json_safe(summary), f, indent=4, ensure_ascii=False)

    logger.info("DONE.")
    logger.info("Outputs saved in: %s", cfg.output_path)


if __name__ == "__main__":
    main()

In [ ]:
import json

paths = {
    "BERT 512": "outputs/06_bert_512_baseline/test_metrics.json",
    "BERT Head+Tail": "outputs/07_bert_head_tail/test_metrics.json",
    "BERT Relevant Chunks": "outputs/08_bert_relevant_chunks/test_metrics.json",
}

for name, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        print(name)
        print(json.load(f))
        print()

OSError: [Errno 107] Transport endpoint is not connected: 'outputs/06_bert_512_baseline/test_metrics.json'

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/cleansteps

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_455/2357226933.py", line 1, in <cell line: 0>
    get_ipython().run_line_magic('cd', '/content/drive/MyDrive/cleansteps')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occu